# Etapa 6: pré-treino geral e ajuste fino com Machado, no Colab

Este notebook pré-treina (pretraining) o GPT médio da etapa 5 (~12 M de parâmetros
(parameters)) no Wikisource em português, **sem Machado**, e depois faz o ajuste fino
(fine-tuning) com o treino de Machado. No fim, compara com o médio "só Machado" da
etapa 5, na mesma validação e em **bits por byte**.

O notebook é fino de propósito: toda a lógica está no pacote `machado_gpt`, que tem
testes. O plano e o porquê de cada escolha estão em `docs/etapas/06-pretreino.md`.

**Antes de começar:** menu *Ambiente de execução → Alterar o tipo de ambiente de
execução → GPU T4*.

**Não salve este notebook com as saídas (outputs) de volta no GitHub:** elas teriam
textos gerados e caminhos do seu Drive.

## 1. Conferir a GPU

**Por quê:** a GPU faz milhares de multiplicações em paralelo. O treino é quase só
multiplicação de matrizes, então ela é dezenas de vezes mais rápida que a CPU da etapa 3.
A T4 tem 16 GB de memória, de sobra para modelos deste tamanho.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Sem GPU: troque o tipo de ambiente de execução para T4"
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name())

## 2. Baixar o código

**Por quê:** o repositório é privado, então o Colab precisa de uma credencial para lê-lo.
Usamos um token do GitHub *fine-grained*, só de leitura e só deste repositório, guardado
nos **Segredos** do Colab (ícone de chave na barra lateral), com o nome `GITHUB_TOKEN`.
O token nunca aparece no notebook nem fica gravado no `.git/config`: ele vai num
cabeçalho (header) passado só para o comando `git clone`.

Preencha `REPO` com `seu-usuario/machado-gpt`. `BRANCH` é a versão do código.

In [ ]:
REPO = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

import base64, os, subprocess
from google.colab import userdata

assert REPO, "Preencha REPO (por exemplo: seu-usuario/machado-gpt)"
ROOT = "/content/machado-gpt"
if not os.path.exists(ROOT):
    auth = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()
    result = subprocess.run(
        ["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}",
         "clone", "--quiet", "--branch", BRANCH, f"https://github.com/{REPO}.git", ROOT],
        capture_output=True, text=True)
    del auth
    assert result.returncode == 0, "git clone falhou: confira REPO, BRANCH e o segredo GITHUB_TOKEN"
%cd {ROOT}
!git log --oneline -1

## 3. Instalar as dependências nas versões exatas

**Por quê:** além do `pdfplumber` (PDFs de Machado), agora precisamos do
`mwparserfromhell`, que analisa o wikitexto do Wikisource. As versões são as do
`uv.lock`: outra versão poderia ler o texto de outro jeito, e o corpus mudaria. O passo
5 confere que não mudou.

In [ ]:
!pip install --quiet pdfplumber==0.11.10 pdfminer.six==20260107 mwparserfromhell==0.7.2
%env PYTHONPATH=/content/machado-gpt/src
import sys
sys.path.insert(0, "/content/machado-gpt/src")

## 4. Regenerar os dois corpora e os símbolos (tokens)

**Por quê:** nenhum corpus entra no Git nem no Drive. Refazemos o caminho inteiro:

1. **Machado** (etapas 1 e 4): download dos PDFs, extração (10 a 20 min), divisão por
   obra e o tokenizador BPE treinado só no treino;
2. **Wikisource** (etapa 6): download do dump fixado (88 MB, conferido pelo SHA-1),
   montagem do corpus com Machado removido (cerca de 20 min) e codificação com o mesmo
   tokenizador.

Tudo é determinístico: o resultado deve ser idêntico, byte a byte, ao da máquina local.

In [ ]:
!python -m machado_gpt.corpus.download

In [ ]:
from pathlib import Path
from machado_gpt.corpus import extract

names = sorted(str(p.relative_to(extract.PDF_DIR)) for p in extract.PDF_DIR.glob("*/*.pdf"))
print(len(names), "PDFs")
extract.main(names)

In [ ]:
!python -m machado_gpt.corpus.build
!python -m machado_gpt.tokenize_corpus

In [ ]:
!python -m machado_gpt.build_general
!python -m machado_gpt.tokenize_general

## 5. Conferir que são os mesmos dados

**Por quê:** "deve ser idêntico" não basta: medimos. O catálogo de Machado, o
`results/04-bpe.json` e o `results/06-wikisource.json` regenerados são comparados com
as versões do Git (SHA-256 do texto de cada obra, do tokenizador e de cada conjunto
codificado). Depois restauramos as versões do Git.

O script de treino confere os hashes de novo e se recusa a treinar se algo diferir.

In [ ]:
import json, subprocess

def git_json(path):
    return json.loads(subprocess.run(["git", "show", f"HEAD:{path}"], capture_output=True, text=True).stdout)

diff = subprocess.run(["git", "diff", "--stat", "data/catalog/"], capture_output=True, text=True).stdout
assert not diff, "o catálogo mudou: a extração no Colab difere da local\n" + diff
new, old = json.load(open("results/04-bpe.json")), git_json("results/04-bpe.json")
assert new["tokenizer_sha256"] == old["tokenizer_sha256"], "tokenizador diferente"
for name in ("train", "val", "test"):
    assert new["splits"][name]["sha256"] == old["splits"][name]["sha256"], f"Machado {name} diferente"
new, old = json.load(open("results/06-wikisource.json")), git_json("results/06-wikisource.json")
assert new["stats"] == old["stats"], "o corpus geral mudou"
for name in ("train", "val"):
    assert new["tokens"]["splits"][name]["sha256"] == old["tokens"]["splits"][name]["sha256"], f"geral {name} diferente"
!git checkout -- results/04-bpe.json results/06-wikisource.json
print("Machado, Wikisource, tokenizador e conjuntos idênticos aos locais")

## 6. Conectar o Google Drive para os checkpoints

**Por quê:** tudo em `/content` some quando a sessão termina. Os checkpoints (`best.pt`
e `last.pt`, para retomar) e uma cópia de cada `results/06-*.json` vão para uma pasta
no seu Drive.

In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/machado-gpt"

def restore(name):
    # Brings a finished phase's results back from Drive after a dropped session.
    copy = f"{CKPT}/{name}/{name}.json"
    if os.path.exists(copy) and not os.path.exists(f"results/{name}.json"):
        shutil.copy(copy, f"results/{name}.json")
    return os.path.exists(f"results/{name}.json")

## 7. Pré-treino: uma volta pelo Wikisource

**O que muda em relação à etapa 5:**

- **uma volta exata:** o texto é cortado em janelas (windows) de 256 símbolos, sem
  sobreposição, embaralhadas; cada uma é lida uma vez (3.254 passos). Na etapa 5 as
  janelas eram sorteadas com reposição, o que numa volta só deixaria ~37% do texto sem
  ler;
- **dropout 0:** numa volta só não há como decorar;
- **sem parada antecipada (early stopping):** a volta vai até o fim.

**O que observar:** a cada 100 passos aparecem três perdas (loss): treino, validação
geral (`val`) e validação de Machado (`machado_val`). Esta última mostra quanto o
modelo "sabe de Machado" sem nunca tê-lo lido. Ela não decide nada.

Cerca de 13 min. Se a sessão cair, rode os passos 1 a 6 e esta célula de novo: o
`--resume` continua do último `last.pt`.

In [ ]:
if restore("06-pretrain-medium"):
    print("pré-treino já concluído (resultado restaurado do Drive)")
else:
    !python -m machado_gpt.train_general --phase pretrain --size medium --out {CKPT}/06-pretrain-medium --resume

## 8. Ajuste fino com o treino de Machado

**O que muda:** partimos dos pesos (weights) do pré-treino (o script confere o SHA-256
do `best.pt` contra o registrado no passo 7), com otimizador (optimizer) novo, taxa de
aprendizado (learning rate) de um terço (2·10⁻⁴) e dropout 0,2. A parada antecipada pela
validação de Machado volta, como na etapa 5.

**O que observar:** a validação de Machado deve começar bem abaixo do ponto de partida
da etapa 5. A validação geral (`general_val`) mostra quanto o modelo esquece do
português geral enquanto aprende Machado.

In [ ]:
if restore("06-finetune-medium"):
    print("ajuste fino já concluído (resultado restaurado do Drive)")
else:
    !python -m machado_gpt.train_general --phase finetune --size medium --init {CKPT}/06-pretrain-medium/best.pt --out {CKPT}/06-finetune-medium --resume

## 9. Comparar com a etapa 5

**Por quê:** a pergunta da etapa é se o pré-treino melhora a validação de Machado.
O gráfico junta as três curvas na validação de Machado: só Machado (etapa 5),
pré-treino (que nunca viu Machado) e ajuste fino (desenhado depois do pré-treino).

In [ ]:
import json
import matplotlib.pyplot as plt

s5 = json.loads(subprocess.run(["git", "show", "HEAD:results/05-gpt-bpe-medium.json"], capture_output=True, text=True).stdout)
pre = json.load(open("results/06-pretrain-medium.json"))
ft = json.load(open("results/06-finetune-medium.json"))
offset = pre["train"]["steps"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([p["step"] for p in s5["history"]], [p["val"] for p in s5["history"]], label="etapa 5: só Machado")
ax.plot([p["step"] for p in pre["history"]], [p["machado_val"] for p in pre["history"]], label="pré-treino (sem Machado)")
ax.plot([offset + p["step"] for p in ft["history"]], [p["val"] for p in ft["history"]], label="ajuste fino")
ax.plot([offset + p["step"] for p in ft["history"]], [p["train"] for p in ft["history"]], "--", label="ajuste fino: treino")
ax.set_xlabel("passo"); ax.set_ylabel("perda na validação de Machado (nats por símbolo)")
ax.set_ylim(top=6); ax.legend(); ax.grid(alpha=0.3)
plt.show()

print(f"{'modelo':34} {'val bits/byte':>14} {'teste bits/byte':>16}")
print(f"{'etapa 5: médio só Machado':34} {s5['val']['bits_per_byte']:14.3f} {s5['test']['bits_per_byte']:16.3f}")
print(f"{'pré-treinado, antes do ajuste':34} {pre['eval']['machado_val']['bits_per_byte']:14.3f}")
print(f"{'pré-treinado + ajuste fino':34} {ft['eval']['machado_val']['bits_per_byte']:14.3f} {ft['eval']['machado_test']['bits_per_byte']:16.3f}")

## 10. Trazer os resultados de volta

**Por quê:** os `results/06-*.json` (métricas, curvas, amostras (samples) e o SHA-256 de
cada `best.pt`) são versionados no Git; os pesos, não. Baixe os dois arquivos e
coloque-os na pasta `results/` do projeto local. Há uma cópia de cada um no Drive.

In [ ]:
from google.colab import files
for name in ("06-pretrain-medium", "06-finetune-medium"):
    files.download(f"results/{name}.json")